# Sonata + linear head — urban tree segmentation

Frozen **Sonata** (self-supervised PTv3) encoder, multi-scale features upcast to full point resolution,
a 2-class linear head trained on **WHU-STree**, then inference on a 100x100 m tile cut from the
13.72 GB campus LAZ.

**Requirements:** GPU accelerator + Internet ON. **Run cells in order** — cell 1 must run before any import.

**Data layout (verified, differs from the README)**

```
whu-stree-nj/<tile>/PCD/<run>.ply     x y z intensity tree label
whu-stree-nj/<tile>/hdi/<run>/traj.csv
whu-stree-nj/reference_data/*.npy     field survey subset, not needed here
```

`tree` = instance id (0 = not a tree), `label` = species id (-1 = not a tree). Binary target is `tree > 0`.
One PLY is ~21.7M points / ~890 MB, which is plenty for a linear probe — we download two.

**Licence:** WHU-STree is CC-BY-NC-ND 3.0, Sonata weights CC-BY-NC 4.0. Research only, no redistribution
of derived copies. Cite the WHU-STree paper.

**Expectations:** Sonata was pretrained mostly on indoor RGB-D. Your data is outdoor MLS with intensity
and no colour. This probe is a *diagnostic*: >0.6 tree IoU means the representation transfers and
fine-tuning is worth it; <0.4 means pretrain Sonata on your own unlabeled scans instead.

**This copy adds the field survey.** Identical pipeline, identical config — the only change is that
`Poderevka-2_final.csv` (4050 GNSS-surveyed tree stems, 187 of them inside this tile) is projected to
UTM 52N and overlaid on the 2D result figure, so the prediction can be read against ground truth
instead of eyeballed. Detected stems draw as hollow black circles, missed stems as blue crosses,
each labelled with its survey id so a suspect tree can be looked up in the CSV.


## 1. Environment guards — run first

`cumm` and `spconv` decide whether to JIT-compile using `pccm.utils.project_is_editable`, which checks
whether a `setup.py` or `.gitignore` exists two directories up from the package — i.e. in
`dist-packages` itself. A stray file there makes every pccm package think it's an editable checkout and
rebuild from source, which fails. Both expose an env var to skip it; the prebuilt `.so` ships in the wheel.

In [ ]:
import os
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'

import sys
sp = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages'
for f in ['setup.py', '.gitignore']:
    p = os.path.join(sp, f)
    if os.path.exists(p):
        os.rename(p, p + '.disabled')
        print(f'renamed stray {f} in dist-packages')
print('jit guards set')

## 2. Install

`spconv-cu126` is the newest variant with cp312 wheels (cu120 stops at cp311; cu128 does not exist).
It runs fine on a CUDA 12.8 runtime via minor-version compatibility. `gdown` needs `-U`: the
preinstalled 5.x caps folder listings at 50 files per folder.

In [ ]:
import subprocess, torch, urllib.request

TORCH_V = torch.__version__.split('+')[0]
CU = (torch.version.cuda or '12.4').replace('.', '')
print('torch', TORCH_V, '| cuda', torch.version.cuda, '| python', sys.version.split()[0])

def sh(cmd):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1200:]); print(r.stderr[-1200:])
    return r.returncode

sh('pip install -q spconv-cu126')

def index_ok(u):
    try:
        return urllib.request.urlopen(u, timeout=15).status == 200
    except Exception:
        return False

idx = f'https://data.pyg.org/whl/torch-{TORCH_V}+cu{CU}.html'
if index_ok(idx):
    sh(f'pip install -q torch-scatter -f {idx}')
else:
    print('no prebuilt torch-scatter, building from source (~10 min)')
    sh('pip install -q --no-build-isolation torch-scatter')

sh('pip install -q -U "gdown>=6.1"')
sh('pip install -q huggingface_hub timm addict laspy[lazrs] plyfile')
sh('pip install -q open3d "numpy<2"')
sh('pip install -q git+https://github.com/facebookresearch/sonata.git')

If this is a fresh session, **restart the kernel now** (numpy was downgraded and gdown upgraded),
then re-run cell 1 and continue from here.

In [ ]:
import os
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'

import numpy as np, torch, cumm, spconv, spconv.pytorch, gdown, laspy
from cumm import core_cc
from tqdm.auto import tqdm

import sonata
print('cumm  ', cumm.__version__, '| prebuilt so:', os.path.basename(core_cc.__file__))
print('spconv', spconv.__version__)
print('gdown ', gdown.__version__)
print('numpy ', np.__version__)
print('sonata ok | cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 3. Config

In [ ]:
from pathlib import Path

WHU_DRIVE_URL = 'https://drive.google.com/drive/folders/18braC_G3RAi5fn8TZNRe-Vt8q1bYRZqk'
WHU_CITY = 'whu-stree-nj'
WHU_MAX_PLY = 2

CAMPUS_LAZ = '/kaggle/input/datasets/samsuperman12/my-private-lidar-dataset/point_cloud_campus_2025-03-14-12-59-27_result.laz'

TILE_SIZE = 100.0
TILE_CENTER = None
CHUNK_SIZE = 25.0
CHUNK_OVERLAP = 3.0

GRID_SIZE = 0.05
TRAIN_TILE_SIZE = 30.0
MAX_TRAIN_TILES = 40
FEAT_SUBSAMPLE = 40000

WORK = Path('/kaggle/working')
RAW = Path('/kaggle/temp')
CACHE = WORK / 'cache'
CACHE.mkdir(parents=True, exist_ok=True)
RAW.mkdir(parents=True, exist_ok=True)

torch.manual_seed(0)
np.random.seed(0)
print('config ok')

## 4. Fetch two PLY scenes

The manifest is cached, so the folder walk runs once. We take `.ply` only — the 12k jpgs and the
trajectory csvs are not used.

In [ ]:
import json, collections

MANIFEST = CACHE / 'drive_manifest.json'
if MANIFEST.exists():
    listing = json.loads(MANIFEST.read_text())
    print('cached manifest:', len(listing), 'files')
else:
    files = gdown.download_folder(url=WHU_DRIVE_URL, skip_download=True, quiet=True)
    listing = [{'id': f.id, 'path': f.path} for f in files]
    MANIFEST.write_text(json.dumps(listing))
    print('enumerated:', len(listing), 'files')

plys = [f for f in listing
        if f['path'].lower().endswith('.ply') and WHU_CITY.lower() in f['path'].lower()]
plys.sort(key=lambda f: f['path'])
print(f'{len(plys)} ply in {WHU_CITY}')

todo = plys[:WHU_MAX_PLY]
WHU_DIR = RAW / 'whu'
WHU_DIR.mkdir(parents=True, exist_ok=True)

got = []
for f in todo:
    parts = Path(f['path']).parts
    tile = next((p for p in parts if p.isdigit()), 'x')
    dst = WHU_DIR / f"{tile}_{Path(f['path']).name}"
    if not dst.exists():
        print('downloading', f['path'])
        gdown.download(id=f['id'], output=str(dst), quiet=False)
    got.append(dst)

for p in got:
    print(f'  {p.name}  {p.stat().st_size/1e6:.0f} MB')

## 5. Load and tile

The grid subsample flattens 3D cell indices to a single int64 key before `np.unique` — `unique(axis=0)`
on 20M+ rows is painfully slow, the flat version is seconds.

In [ ]:
from plyfile import PlyData

def read_ply_scene(path):
    v = PlyData.read(str(path))['vertex'].data
    xyz = np.column_stack([v['x'], v['y'], v['z']]).astype(np.float64)
    inten = np.asarray(v['intensity'], dtype=np.float32)
    inst = np.asarray(v['tree'], dtype=np.int64)
    species = np.asarray(v['label'], dtype=np.int64) if 'label' in v.dtype.names else None
    return xyz, inten, inst, species


def grid_subsample(xyz, feats, size):
    k = np.floor(xyz / size).astype(np.int64)
    k -= k.min(0)
    dim = k.max(0) + 1
    flat = k[:, 0] + k[:, 1] * dim[0] + k[:, 2] * dim[0] * dim[1]
    _, idx = np.unique(flat, return_index=True)
    idx.sort()
    return xyz[idx], [f[idx] for f in feats]


def tile_scene(xyz, inten, lab, tile, min_pts=8000, max_tiles=999):
    o = xyz[:, :2].min(0)
    ij = np.floor((xyz[:, :2] - o) / tile).astype(np.int64)
    keys, inv = np.unique(ij, axis=0, return_inverse=True)
    out = []
    for k in np.random.permutation(len(keys)):
        m = inv == k
        if m.sum() < min_pts:
            continue
        out.append((xyz[m], inten[m], lab[m]))
        if len(out) >= max_tiles:
            break
    return out


train_tiles = []
per_file = MAX_TRAIN_TILES // max(1, len(got)) + 2

for p in got:
    xyz, inten, inst, species = read_ply_scene(p)
    lab = (inst > 0).astype(np.int64)

    if species is not None:
        agree = ((species >= 0) == (lab == 1)).mean()
        print(f'{p.name}: {len(xyz):,} pts | tree {lab.mean():.3f} | '
              f'instances {len(np.unique(inst))-1} | label/tree agreement {agree:.4f}')
    else:
        print(f'{p.name}: {len(xyz):,} pts | tree {lab.mean():.3f}')

    xyz, (inten, lab) = grid_subsample(xyz, [inten, lab], GRID_SIZE)
    print(f'  after {GRID_SIZE} m grid: {len(xyz):,}')
    ts = tile_scene(xyz, inten, lab, TRAIN_TILE_SIZE, max_tiles=per_file)
    train_tiles += ts
    print(f'  -> {len(ts)} tiles')
    del xyz, inten, lab, inst, species

train_tiles = train_tiles[:MAX_TRAIN_TILES]
assert train_tiles
frac = np.mean([t[2].mean() for t in train_tiles])
print(f'\n{len(train_tiles)} tiles | mean tree fraction {frac:.3f}')

rng = np.random.default_rng(0)
order = rng.permutation(len(train_tiles))
train_tiles = [train_tiles[i] for i in order]

split = int(len(train_tiles) * 0.8)
tiles_tr, tiles_va = train_tiles[:split], train_tiles[split:]
print(f'train {len(tiles_tr)} | val {len(tiles_va)}')
print(f'tree frac — train {np.mean([t[2].mean() for t in tiles_tr]):.4f} '
      f'| val {np.mean([t[2].mean() for t in tiles_va]):.4f}')

## 6. Sonata input

Sonata wants coord + colour + normal (9 channels — it is an indoor RGB-D model at heart). MLS has
neither colour nor normals, so intensity is mapped to greyscale and normals are estimated with Open3D.
This substitution *is* the domain gap, made concrete.

In [ ]:
import open3d as o3d

def estimate_normals(xyz, radius=0.5, max_nn=30):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=radius, max_nn=max_nn))
    pcd.normalize_normals()
    return np.asarray(pcd.normals, dtype=np.float32)


USE_INTENSITY = False

def make_point(xyz, inten, segment=None):
    xyz = np.asarray(xyz, dtype=np.float64)
    xyz = xyz - xyz.mean(0, keepdims=True)
    xyz = xyz.astype(np.float32)
    if USE_INTENSITY:
        color = np.repeat((inten.astype(np.float32) / 65535.0)[:, None], 3, axis=1) * 255.0
    else:
        color = np.full((len(xyz), 3), 127.5, dtype=np.float32)
    d = dict(coord=xyz, color=color.astype(np.float32),
             normal=estimate_normals(xyz.astype(np.float64)))
    if segment is not None:
        d['segment'] = segment.astype(np.int64)
    return d

In [ ]:
custom_config = dict(enc_patch_size=[1024] * 5, enable_flash=False)
model = sonata.load('sonata', repo_id='facebook/sonata', custom_config=custom_config).cuda()
model.eval()

transform = sonata.transform.Compose([
    dict(type='CenterShift', apply_z=True),
    dict(type='GridSample', grid_size=GRID_SIZE, hash_type='fnv', mode='train',
         return_grid_coord=True, return_inverse=True),
    dict(type='NormalizeColor'),
    dict(type='ToTensor'),
    dict(type='Collect', keys=('coord', 'grid_coord', 'color', 'inverse'),
         feat_keys=('coord', 'color', 'normal')),
])
print('model ready')

### Upcast

Sonata is decoder-free: it returns an encoder ladder, not one vector per point. Each pooling step
recorded which children merged into which parent, so we read that mapping backwards. Two stages get
concatenated (deep + shallow), the rest propagate. `point.inverse` then maps grid-sampled points back
to the original cloud.

In [ ]:
@torch.no_grad()
def sonata_feat(pd):
    point = transform(dict(pd))
    for k in point.keys():
        if isinstance(point[k], torch.Tensor):
            point[k] = point[k].cuda(non_blocking=True)
    point = model(point)
    for _ in range(2):
        parent = point.pop('pooling_parent')
        inv = point.pop('pooling_inverse')
        parent.feat = torch.cat([parent.feat, point.feat[inv]], dim=-1)
        point = parent
    while 'pooling_parent' in point.keys():
        parent = point.pop('pooling_parent')
        inv = point.pop('pooling_inverse')
        parent.feat = point.feat[inv]
        point = parent
    return point.feat[point.inverse].float().cpu()


f = sonata_feat(make_point(*tiles_tr[0][:2], segment=tiles_tr[0][2]))
FEAT_DIM = f.shape[1]
print('feature dim', FEAT_DIM, '| points', f.shape[0])

## 7. Extract features

In [ ]:
import gc

def gpu_free():
    gc.collect()
    torch.cuda.empty_cache()

def try_feat(xyz, inten, lab):
    try:
        return sonata_feat(make_point(xyz, inten, segment=lab)).numpy()
    except torch.cuda.OutOfMemoryError:
        return None

def feat_recursive(xyz, inten, lab, depth=0):
    f = try_feat(xyz, inten, lab)
    if f is not None:
        return [(f, lab)]
    gpu_free()
    if depth >= 6 or len(xyz) < 2000:
        print(f'  dropped {len(xyz):,} pts')
        return []
    ax = 0 if xyz[:, 0].ptp() >= xyz[:, 1].ptp() else 1
    mid = np.median(xyz[:, ax])
    m = xyz[:, ax] < mid
    if m.all() or (~m).all():
        return []
    return (feat_recursive(xyz[m], inten[m], lab[m], depth + 1) +
            feat_recursive(xyz[~m], inten[~m], lab[~m], depth + 1))

def build_set(tiles, sub=FEAT_SUBSAMPLE):
    X, Y = [], []
    for xyz, inten, lab in tqdm(tiles):
        for f, y in feat_recursive(xyz, inten, lab):
            n = min(sub, len(y))
            sel = np.random.choice(len(y), n, replace=False)
            X.append(f[sel]); Y.append(y[sel])
        gpu_free()
    return np.concatenate(X), np.concatenate(Y)

Xtr, Ytr = build_set(tiles_tr)
Xva, Yva = build_set(tiles_va)
print('train', Xtr.shape, '| tree frac', round(float(Ytr.mean()), 4))
print('val  ', Xva.shape, '| tree frac', round(float(Yva.mean()), 4))

## 8. Linear head

Weighted CE and model selection on tree IoU. Unweighted CE on a minority class gives a 90%-accurate
model that predicts background everywhere.

In [ ]:
import torch.nn as nn

class LinearHead(nn.Module):
    def __init__(self, dim, n=2):
        super().__init__()
        self.norm = nn.BatchNorm1d(dim)
        self.fc = nn.Linear(dim, n)
    def forward(self, x):
        return self.fc(self.norm(x))


def iou_tree(p, y):
    inter = ((p == 1) & (y == 1)).sum()
    union = ((p == 1) | (y == 1)).sum()
    return float(inter) / max(1, int(union))


head = LinearHead(FEAT_DIM).cuda()
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)

pos = float(Ytr.mean())
w = torch.tensor([pos, 1 - pos]).float().cuda()
w = w / w.sum() * 2
print('class weights (bg, tree):', [round(x, 3) for x in w.tolist()])
crit = nn.CrossEntropyLoss(weight=w)

Xtr_t, Ytr_t = torch.from_numpy(Xtr).float(), torch.from_numpy(Ytr).long()
Xva_t, Yva_t = torch.from_numpy(Xva).float().cuda(), torch.from_numpy(Yva).long().cuda()

best, best_state = 0.0, None
for ep in range(30):
    head.train()
    perm = torch.randperm(len(Xtr_t))
    tot = 0.0
    for i in range(0, len(perm), 8192):
        idx = perm[i:i + 8192]
        xb, yb = Xtr_t[idx].cuda(), Ytr_t[idx].cuda()
        opt.zero_grad()
        loss = crit(head(xb), yb)
        loss.backward(); opt.step()
        tot += loss.item() * len(idx)
    head.eval()
    with torch.no_grad():
        pv = head(Xva_t).argmax(1)
    iou = iou_tree(pv.cpu().numpy(), Yva)
    if iou > best:
        best, best_state = iou, {k: v.clone() for k, v in head.state_dict().items()}
    if ep % 5 == 0 or ep == 29:
        acc = (pv == Yva_t).float().mean().item()
        print(f'ep {ep:2d} | loss {tot/len(perm):.4f} | acc {acc:.4f} | tree IoU {iou:.4f}')

head.load_state_dict(best_state)
print(f'\nbest val tree IoU {best:.4f}')
torch.save({'state_dict': best_state, 'feat_dim': FEAT_DIM, 'iou': best}, WORK / 'linear_head.pth')

## 9. Cut the campus tile

In [ ]:
with laspy.open(CAMPUS_LAZ) as f:
    h = f.header
    print('points:', f'{h.point_count:,}')
    print('mins  :', np.round(h.mins, 2))
    print('maxs  :', np.round(h.maxs, 2))
    print('extent:', np.round((h.maxs - h.mins)[:2], 1), 'm')
    print('dims  :', [d.name for d in h.point_format.dimensions])
    MINS, MAXS = np.array(h.mins), np.array(h.maxs)

if TILE_CENTER is None:
    TILE_CENTER = ((MINS[:2] + MAXS[:2]) / 2).tolist()
half = TILE_SIZE / 2
BBOX = (TILE_CENTER[0] - half, TILE_CENTER[0] + half,
        TILE_CENTER[1] - half, TILE_CENTER[1] + half)
print('\ncenter:', np.round(TILE_CENTER, 2), '| bbox:', np.round(BBOX, 2))

## 9b. Ground truth — surveyed tree stems

`Poderevka-2_final.csv` is a per-tree field survey of the same site: `id, lat, lon, elevation`,
no header, WGS84 geographic. Rows whose first field is `base_N` are not trees — they mark a change
of GNSS reference station mid-survey and are skipped.

The LAZ is in projected coordinates (easting ~736 km, northing ~4768 km = **UTM 52N**), so the survey
has to be projected to match. The Transverse Mercator forward is inlined rather than pulled from
pyproj: `numpy<2` is installed above, and binary deps compiled against numpy 2 are a coin flip after
that downgrade.

This is a **stem map** — position and ground elevation only, no crown extent, DBH, height or species.
So it supports detection recall (does each surveyed stem have predicted tree points over it) but not
point-wise IoU. Surveyed z sits ~0.45 m below the LiDAR ground return, consistent across the tile.


In [ ]:
import csv, re, math

UTM_ZONE = 52          # WGS84 / UTM 52N — matches the campus LAZ easting/northing
GT_RADIUS = 2.0        # m, horizontal search radius around a stem
GT_MIN_PTS = 20        # predicted tree points inside that radius to call a stem detected


def ll_to_utm(lat, lon, zone=UTM_ZONE):
    a, f, k0 = 6378137.0, 1 / 298.257223563, 0.9996
    e2 = 2 * f - f * f
    ep2 = e2 / (1 - e2)
    lat = np.radians(lat)
    dl = np.radians(lon) - math.radians(zone * 6 - 183)
    N = a / np.sqrt(1 - e2 * np.sin(lat) ** 2)
    T = np.tan(lat) ** 2
    C = ep2 * np.cos(lat) ** 2
    A = dl * np.cos(lat)
    M = a * ((1 - e2 / 4 - 3 * e2 ** 2 / 64 - 5 * e2 ** 3 / 256) * lat
             - (3 * e2 / 8 + 3 * e2 ** 2 / 32 + 45 * e2 ** 3 / 1024) * np.sin(2 * lat)
             + (15 * e2 ** 2 / 256 + 45 * e2 ** 3 / 1024) * np.sin(4 * lat)
             - (35 * e2 ** 3 / 3072) * np.sin(6 * lat))
    x = k0 * N * (A + (1 - T + C) * A ** 3 / 6
                  + (5 - 18 * T + T * T + 72 * C - 58 * ep2) * A ** 5 / 120) + 500000
    y = k0 * (M + N * np.tan(lat) * (A * A / 2
              + (5 - T + 9 * C + 4 * C * C) * A ** 4 / 24
              + (61 - 58 * T + T * T + 600 * C - 330 * ep2) * A ** 6 / 720))
    return x, y


def load_stems(path):
    """id, lat, lon, elev. Skips base_N reference-station rows. Fields carry trailing tabs."""
    sid, lat, lon, elev, base, cur = [], [], [], [], [], 'base_1'
    with open(path) as fh:
        for row in csv.reader(fh):
            if len(row) < 4:
                continue
            if not re.fullmatch(r'-?\d+', row[0].strip()):
                cur = row[0].strip()
                continue
            sid.append(int(row[0]))
            lat.append(float(row[1]))
            lon.append(float(row[2]))
            elev.append(float(row[3]))
            base.append(cur)
    return (np.array(sid), np.array(lat), np.array(lon),
            np.array(elev, np.float64), np.array(base))


GT_CSV = None
_root = Path('/kaggle/input')
if _root.exists():
    _hits = sorted(_root.rglob('*oderevka*.csv')) + sorted(_root.rglob('*odereopen*.csv'))
    if _hits:
        GT_CSV = str(_hits[0])

if GT_CSV is None:
    print('!! no survey csv found under /kaggle/input')
    print('   attach the stem-map dataset to this notebook; markers will be skipped for now')
    gt_xy = np.zeros((0, 2))
    gt_z = np.zeros(0)
    gt_id = np.zeros(0, np.int64)
    gt_base = np.zeros(0, dtype=object)
else:
    print('ground truth:', GT_CSV)
    s_id, s_lat, s_lon, s_z, s_base = load_stems(GT_CSV)
    s_x, s_y = ll_to_utm(s_lat, s_lon)
    print(f'{len(s_id):,} surveyed stems | {len(set(s_base))} RTK sessions')
    print(f'  survey extent  x {s_x.min():.0f}..{s_x.max():.0f}  y {s_y.min():.0f}..{s_y.max():.0f}')
    print(f'  tile bbox      x {BBOX[0]:.0f}..{BBOX[1]:.0f}  y {BBOX[2]:.0f}..{BBOX[3]:.0f}')

    m = ((s_x >= BBOX[0]) & (s_x <= BBOX[1]) & (s_y >= BBOX[2]) & (s_y <= BBOX[3]))
    gt_xy = np.column_stack([s_x[m], s_y[m]])
    gt_z = s_z[m]
    gt_id = s_id[m]
    gt_base = s_base[m]
    print(f'\n  {m.sum()} stems inside the tile')
    if m.sum() == 0:
        d = math.hypot(s_x.mean() - np.mean(BBOX[:2]), s_y.mean() - np.mean(BBOX[2:]))
        print(f'  !! survey centroid is {d:,.0f} m from the tile centre — wrong UTM_ZONE, '
              f'wrong TILE_CENTER, or the LAZ is on a different CRS')
    else:
        print(f'  elevation {gt_z.min():.2f}..{gt_z.max():.2f} m | sessions {sorted(set(gt_base))}')


In [ ]:
def extract_bbox(path, bbox, grid=GRID_SIZE, chunk=8_000_000):
    x0, x1, y0, y1 = bbox
    xs, ys, zs, its = [], [], [], []
    kept = 0
    with laspy.open(path) as f:
        has_i = 'intensity' in [d.name for d in f.header.point_format.dimensions]
        pbar = tqdm(total=f.header.point_count, unit='pts')
        for pts in f.chunk_iterator(chunk):
            pbar.update(len(pts.x))
            X, Y = np.asarray(pts.x), np.asarray(pts.y)
            m = (X >= x0) & (X < x1) & (Y >= y0) & (Y < y1)
            if not m.any():
                continue
            xs.append(X[m]); ys.append(Y[m]); zs.append(np.asarray(pts.z)[m])
            its.append(np.asarray(pts.intensity, np.float32)[m] if has_i
                       else np.zeros(int(m.sum()), np.float32))
            kept += int(m.sum())
        pbar.close()
    if not kept:
        raise RuntimeError('empty tile — adjust TILE_CENTER')
    xyz = np.column_stack([np.concatenate(xs), np.concatenate(ys), np.concatenate(zs)])
    inten = np.concatenate(its)
    print(f'\nkept {kept:,}')
    xyz, (inten,) = grid_subsample(xyz, [inten], grid)
    print(f'after grid: {len(xyz):,}')
    return xyz, inten

cf = CACHE / f'tile_{int(TILE_CENTER[0])}_{int(TILE_CENTER[1])}_{int(TILE_SIZE)}.npz'
if cf.exists():
    d = np.load(cf)
    tile_xyz, tile_int = d['xyz'], d['inten']
    print('cached tile:', tile_xyz.shape)
else:
    tile_xyz, tile_int = extract_bbox(CAMPUS_LAZ, BBOX)
    np.savez_compressed(cf, xyz=tile_xyz, inten=tile_int)
print('z range:', round(float(tile_xyz[:, 2].min()), 2), round(float(tile_xyz[:, 2].max()), 2))

In [ ]:
def occupied_density(xyz, cell=1.0):
    k = np.floor(xyz[:, :2] / cell).astype(np.int64)
    _, cnt = np.unique(k, axis=0, return_counts=True)
    return float(np.median(cnt)) / (cell ** 2)

whu_d = np.median([occupied_density(t[0]) for t in tiles_tr])
camp_d = occupied_density(tile_xyz)
print(f'occupied density — WHU {whu_d:.0f} pts/m2 | campus {camp_d:.0f} pts/m2')
print(f'ratio {camp_d/whu_d:.1f}x')

GRID_CAMPUS = GRID_SIZE * np.sqrt(camp_d / whu_d)
print(f'\nGRID_SIZE {GRID_SIZE} -> GRID_CAMPUS {GRID_CAMPUS:.3f} m')

tile_xyz_m, (tile_int_m,) = grid_subsample(tile_xyz, [tile_int], GRID_CAMPUS)
print(f'{len(tile_xyz):,} -> {len(tile_xyz_m):,} pts')
print(f'matched density: {occupied_density(tile_xyz_m):.0f} pts/m2 (target {whu_d:.0f})')

tile_xyz, tile_int = tile_xyz_m, tile_int_m

## 10. Inference

In [ ]:
def chunk_bboxes(xyz, size, ov):
    x0, y0 = xyz[:, 0].min(), xyz[:, 1].min()
    x1, y1 = xyz[:, 0].max(), xyz[:, 1].max()
    step, out, x = size - ov, [], x0
    while x < x1:
        y = y0
        while y < y1:
            out.append((x, min(x + size, x1 + 1e-6), y, min(y + size, y1 + 1e-6)))
            y += step
        x += step
    return out


import gc

def gpu_free():
    gc.collect()
    torch.cuda.empty_cache()

def try_infer(idx):
    try:
        feat = sonata_feat(make_point(tile_xyz[idx], tile_int[idx])).cuda()
        with torch.no_grad():
            lg = head(feat).cpu().numpy()
        del feat
        return lg
    except torch.cuda.OutOfMemoryError:
        return None

def infer_box(idx, depth=0):
    if len(idx) < 500:
        return
    lg = try_infer(idx)
    if lg is not None:
        lsum[idx] += lg
        lcnt[idx] += 1
        return
    gpu_free()
    if depth >= 6:
        print(f'  giving up on {len(idx):,} pts | '
              f'alloc {torch.cuda.memory_allocated()/1e9:.2f} GB '
              f'reserved {torch.cuda.memory_reserved()/1e9:.2f} GB')
        return
    pts = tile_xyz[idx]
    ax = 0 if pts[:, 0].ptp() >= pts[:, 1].ptp() else 1
    mid = np.median(pts[:, ax])
    left, right = idx[pts[:, ax] < mid], idx[pts[:, ax] >= mid]
    if len(left) == 0 or len(right) == 0:
        return
    infer_box(left, depth + 1)
    infer_box(right, depth + 1)


head.eval()
lsum = np.zeros((len(tile_xyz), 2), np.float32)
lcnt = np.zeros(len(tile_xyz), np.float32)
boxes = chunk_bboxes(tile_xyz, CHUNK_SIZE, CHUNK_OVERLAP)
print(len(boxes), 'chunks')

for bx0, bx1, by0, by1 in tqdm(boxes):
    m = ((tile_xyz[:, 0] >= bx0) & (tile_xyz[:, 0] < bx1) &
         (tile_xyz[:, 1] >= by0) & (tile_xyz[:, 1] < by1))
    infer_box(np.where(m)[0])

cov = lcnt > 0
pred = np.full(len(tile_xyz), -1, np.int64)
pred[cov] = lsum[cov].argmax(1)
probs = np.full(len(tile_xyz), np.nan, np.float32)
probs[cov] = torch.softmax(torch.from_numpy(lsum[cov] / lcnt[cov, None]), 1)[:, 1].numpy()

print(f'\ncovered {cov.mean()*100:.1f}%')
print(f'tree within covered: {(pred[cov] == 1).mean()*100:.2f}%')
print(f'tree overall: {(pred == 1).mean()*100:.2f}%')

In [ ]:
print('FEAT_DIM      :', FEAT_DIM)
print('GPU           :', torch.cuda.get_device_name(0),
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('alloc / reserved:', f'{torch.cuda.memory_allocated()/1e9:.2f} /',
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB')

counts = []
for bx0, bx1, by0, by1 in boxes:
    m = ((tile_xyz[:, 0] >= bx0) & (tile_xyz[:, 0] < bx1) &
         (tile_xyz[:, 1] >= by0) & (tile_xyz[:, 1] < by1))
    counts.append(int(m.sum()))
counts = np.array(counts)
print(f'\ncampus chunk pts: min {counts.min():,} med {int(np.median(counts)):,} max {counts.max():,}')

whu = np.array([len(t[0]) for t in tiles_tr])
print(f'WHU tile pts    : min {whu.min():,} med {int(np.median(whu)):,} max {whu.max():,}')
print(f'\ncampus density  : {counts.max()/(CHUNK_SIZE**2):.0f} pts/m2')
print(f'WHU density     : {np.median(whu)/(TRAIN_TILE_SIZE**2):.0f} pts/m2')

## 11. Results

Judge the top-down view first: urban trees should be discrete blobs along paths, not smears across
building walls. Facades lighting up = the indoor-pretraining domain gap.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D

# --- match each surveyed stem against the prediction --------------------------
# x-sorted band slice keeps this exact without pulling in scipy (see numpy<2 note above)
tm = pred == 1
if len(gt_xy):
    _t = tile_xyz[tm][:, :2]
    _t = _t[np.argsort(_t[:, 0])]
    gt_n = np.empty(len(gt_xy), np.int64)
    for _i, (_gx, _gy) in enumerate(gt_xy):
        _lo, _hi = np.searchsorted(_t[:, 0], [_gx - GT_RADIUS, _gx + GT_RADIUS])
        _s = _t[_lo:_hi]
        gt_n[_i] = int((np.hypot(_s[:, 0] - _gx, _s[:, 1] - _gy) <= GT_RADIUS).sum())
    gt_hit = gt_n >= GT_MIN_PTS
    print(f'stem detection: {gt_hit.sum()}/{len(gt_hit)} ({gt_hit.mean() * 100:.1f}%) '
          f'have >={GT_MIN_PTS} predicted tree pts within {GT_RADIUS} m')
    print(f'  missed stems: {list(gt_id[~gt_hit])}')
else:
    gt_n, gt_hit = np.zeros(0, np.int64), np.zeros(0, bool)

HIT = dict(marker='o', s=46, facecolors='none', edgecolors='#111111', linewidths=0.9)
MISS = dict(marker='X', s=52, c='#0033CC', edgecolors='white', linewidths=0.5)

GT_LABELS = True       # annotate each stem with its survey id
GT_LABEL_SIZE = 4.5    # pt. 189 stems in 100 m is dense — zoom the png to read them
_HALO = [pe.withStroke(linewidth=1.4, foreground='white')]


def mark(ax, x, y, hit, ids):
    """Overlay surveyed stems: hollow black = detected, blue X = missed, id alongside."""
    if not len(x):
        return
    ax.scatter(x[hit], y[hit], zorder=5, **HIT)
    ax.scatter(x[~hit], y[~hit], zorder=6, **MISS)
    if not GT_LABELS:
        return
    # offset in points, not data units, so labels hold their spacing on any axis scale
    for _px, _py, _h, _id in zip(x, y, hit, ids):
        ax.annotate(str(_id), (_px, _py), textcoords='offset points', xytext=(4.5, 3.0),
                    fontsize=GT_LABEL_SIZE, color='#111111' if _h else '#0033CC',
                    path_effects=_HALO, zorder=8, clip_on=True)


cmap = ListedColormap(['#B4B2A9', '#1D9E75'])
fig = plt.figure(figsize=(16, 13))
s = np.random.choice(len(tile_xyz), min(400_000, len(tile_xyz)), replace=False)

ax = fig.add_subplot(2, 2, 1)
ax.scatter(tile_xyz[s, 0], tile_xyz[s, 1], c=pred[s], cmap=cmap, s=0.12, marker='.')
mark(ax, gt_xy[:, 0], gt_xy[:, 1], gt_hit, gt_id)
ax.set_title(f'Top-down | {tm.sum():,} tree pts ({tm.mean()*100:.1f}%)'
             + (f' | {gt_hit.sum()}/{len(gt_hit)} stems hit' if len(gt_hit) else ''))
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_aspect('equal')
if len(gt_hit):
    ax.legend(handles=[
        Line2D([], [], ls='', marker='o', mfc='none', mec='#111111', label='surveyed stem — detected'),
        Line2D([], [], ls='', marker='X', color='#0033CC', mec='white', label='surveyed stem — missed'),
    ], loc='upper right', fontsize=8, framealpha=0.9)

ax = fig.add_subplot(2, 2, 2)
sc = ax.scatter(tile_xyz[s, 0], tile_xyz[s, 1], c=probs[s], cmap='RdYlGn',
                s=0.12, marker='.', vmin=0, vmax=1)
mark(ax, gt_xy[:, 0], gt_xy[:, 1], gt_hit, gt_id)
ax.set_title('Tree probability'); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_aspect('equal'); plt.colorbar(sc, ax=ax, fraction=0.046)

ax = fig.add_subplot(2, 2, 3)
ymid = tile_xyz[:, 1].mean()
b = np.where(np.abs(tile_xyz[:, 1] - ymid) < 8)[0]
if len(b) > 200_000:
    b = np.random.choice(b, 200_000, replace=False)
ax.scatter(tile_xyz[b, 0], tile_xyz[b, 2], c=pred[b], cmap=cmap, s=0.2, marker='.')
if len(gt_xy):
    # stems in the same 16 m slice, drawn at their surveyed ground elevation
    sl = np.abs(gt_xy[:, 1] - ymid) < 8
    for _x, _z, _h in zip(gt_xy[sl, 0], gt_z[sl], gt_hit[sl]):
        ax.plot([_x, _x], [_z, _z + 3], lw=0.7, alpha=0.75,
                color='#111111' if _h else '#0033CC', zorder=5)
    mark(ax, gt_xy[sl, 0], gt_z[sl], gt_hit[sl], gt_id[sl])
    ax.set_title(f'Side view (16 m slice) | {sl.sum()} stems')
else:
    ax.set_title('Side view (16 m slice)')
ax.set_xlabel('x (m)'); ax.set_ylabel('z (m)'); ax.set_aspect('equal')

ax = fig.add_subplot(2, 2, 4)
z0 = tile_xyz[:, 2].min()
bins = np.linspace(0, min(30, tile_xyz[:, 2].max() - z0), 60)
ax.hist(tile_xyz[~tm, 2] - z0, bins=bins, alpha=0.65, color='#B4B2A9',
        label='background', orientation='horizontal')
ax.hist(tile_xyz[tm, 2] - z0, bins=bins, alpha=0.8, color='#1D9E75',
        label='tree', orientation='horizontal')
ax.set_xscale('log'); ax.set_ylabel('height above min z (m)')
ax.set_xlabel('points (log)'); ax.set_title('Height profile'); ax.legend()

plt.tight_layout()
plt.savefig(WORK / 'tree_segmentation.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
fig = plt.figure(figsize=(13, 10))
ax = fig.add_subplot(111, projection='3d')
s3 = np.random.choice(len(tile_xyz), min(150_000, len(tile_xyz)), replace=False)
ax.scatter(tile_xyz[s3, 0], tile_xyz[s3, 1], tile_xyz[s3, 2],
           c=pred[s3], cmap=cmap, s=0.4, marker='.', depthshade=False)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_zlabel('z (m)')
ax.set_title(f'{TILE_SIZE:.0f}x{TILE_SIZE:.0f} m campus tile — Sonata + linear head')
ax.view_init(elev=22, azim=-60)
try:
    ax.set_box_aspect((1, 1, 0.35))
except Exception:
    pass
plt.tight_layout()
plt.savefig(WORK / 'tree_segmentation_3d.png', dpi=130, bbox_inches='tight')
plt.show()

### Export

Tree points get classification 5 (ASPRS high vegetation) so CloudCompare colours them correctly.
`tree_prob` is an extra dimension — threshold it yourself to trade precision against recall.

In [ ]:
h = laspy.LasHeader(point_format=3, version='1.2')
h.offsets = tile_xyz.min(0)
h.scales = [0.001, 0.001, 0.001]
out = laspy.LasData(h)
out.x, out.y, out.z = tile_xyz[:, 0], tile_xyz[:, 1], tile_xyz[:, 2]
out.intensity = np.clip(tile_int, 0, 65535).astype(np.uint16)
out.classification = np.where(pred == 1, 5, 1).astype(np.uint8)
out.add_extra_dim(laspy.ExtraBytesParams(name='tree_prob', type=np.float32))
out.tree_prob = probs
out.write(str(WORK / 'campus_tile_segmented.laz'))

np.savez_compressed(WORK / 'campus_tile_pred.npz',
                    xyz=tile_xyz, intensity=tile_int, pred=pred, prob=probs)

for f in sorted(WORK.glob('*')):
    if f.is_file():
        print(f'  {f.name}  {f.stat().st_size/1e6:.1f} MB')
print(f'\nval tree IoU (WHU-STree {WHU_CITY}): {best:.4f}')
print(f'campus tile tree fraction: {pred.mean()*100:.2f}%')

## Next steps

1. **Look at it.** Open `campus_tile_segmented.laz` in CloudCompare, filter to class 5. Thirty seconds
   of eyeballing beats the IoU number, which was measured on Nanjing street MLS, not your campus.
2. **Unfreeze.** If the probe holds up, unfreeze the last encoder stage at LR ~1e-5 with the head at
   1e-3, then full fine-tune with layer-wise LR decay.
3. **Use `label` too.** The PLY carries species ids. Once binary works, the same features support a
   19-way species head for free.
4. **Label your own.** A few hundred corrected tiles from your backpack scans beat any amount of head
   tuning on frozen cross-domain features. Use these predictions as pre-labels.
5. **Add Lovász** once fine-tuning end to end — it optimizes IoU directly.